# Reading Between the Ratings: Predicting Dissatisfied Patient Drug Reviews
**INFO 4360 NLP for Real-World Problems | Phase 3 Final Report | Liz (solo project)**

## Problem Statement
A patient-experience manager at an online pharmacy or health-information site receives thousands of patient drug reviews and cannot read them all, yet the unhappiest reviews (rating of 4 out of 10 or lower) are the ones that signal churn, poor adherence and possible safety problems. This project builds a text classifier on the Drugs.com reviews from the UCI Machine Learning Repository that flags likely-dissatisfied reviews from their wording plus the drug and condition, so the manager gets a ranked queue for outreach, pharmacist follow-up or safety escalation instead of reading everything. A supporting Claude step labels the main reason behind a sample of dissatisfied reviews so the queue comes with context, and every Claude label is hand-checked before use.

**Path A (Prediction).** Target: `dissatisfied = 1` when `rating <= 4`.

In [ ]:
# ---- Setup -------------------------------------------------------------
import os, sys, re, json, html, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.dummy import DummyClassifier
from sklearn.metrics import precision_recall_curve, roc_auc_score, cohen_kappa_score
from scipy.stats import spearmanr
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))                       # lets us import drug_review_models.py from the repo root
DATA = Path(os.environ.get("DRUG_DATA_DIR", ROOT / "data"))
OUT = ROOT / "results"; OUT.mkdir(exist_ok=True)
try:
    from dotenv import load_dotenv; load_dotenv(ROOT / ".env")   # ANTHROPIC_API_KEY lives in .env (git-ignored)
except ImportError:
    pass

from drug_review_models import (load_raw, clean, add_features, feature_lists,
                                build_lr, build_gbm, evaluate, lr_coefficients, RANDOM_STATE)

# ---- Settings (change here only) ---------------------------------------
N_TRAIN = None            # None = full training set; e.g. 30000 for a quick test run
RELABEL = False           # True = call the Claude API again; False = reuse data/claude_labels.csv
N_LABEL = 150             # dissatisfied test reviews sent to Claude
N_HANDCHECK = 30          # random reviews hand-checked against Claude
CLAUDE_MODEL = "claude-haiku-5-5"
PRICE_IN, PRICE_OUT = 0.10, 0.50    # USD per million tokens (Haiku 5.5, prompts up to 100K tokens)
PROMPT_VERSION = "v1"     # bump to "v2" if you revise the prompt after the hand-check

def find(prefix):
    hits = sorted(DATA.glob(f"{prefix}.*"))
    if not hits:
        raise FileNotFoundError(f"Put {prefix}.tsv (or .csv) in {DATA}. See README, step 4.")
    return hits[0]
TRAIN_FILE, TEST_FILE = find("drugsComTrain_raw"), find("drugsComTest_raw")
print("Train:", TRAIN_FILE.name, "| Test:", TEST_FILE.name)

## 1. Data and Cleaning
The data are 215,063 Drugs.com reviews with `drugName`, `condition`, `review`, `rating` (1 to 10), `date` and `usefulCount`, split by the source into a train and a test file. Phase 2 found that about 80% of rows share exact text with another row and that many texts appear in both files, which would inflate test scores. So cleaning here goes one step further than Phase 2: **duplicates are removed within each file and any test review whose text also appears in train is dropped.**

In [ ]:
def parse_dates(s):
    """UCI dates look like '20-May-12'; Kaggle copies look like 'May 20, 2012'. Try each format."""
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for fmt in ("%d-%b-%y", "%B %d, %Y", "%b %d, %Y"):
        out = out.fillna(pd.to_datetime(s, format=fmt, errors="coerce").astype("datetime64[ns]"))
    return out

def prep(raw):
    raw = raw.copy()
    raw["date"] = parse_dates(raw["date"].astype(str)).dt.strftime("%Y-%m-%d")
    return clean(raw)                     # HTML decode, scraped-condition artifacts, missing values, duplicate texts

raw_tr, raw_te = load_raw(TRAIN_FILE), load_raw(TEST_FILE)
tr_clean, te_clean = prep(raw_tr), prep(raw_te)

overlap_mask = te_clean["review"].isin(set(tr_clean["review"]))
te_clean = te_clean[~overlap_mask]

train, test = add_features(tr_clean), add_features(te_clean)
if N_TRAIN and N_TRAIN < len(train):
    train = train.sample(N_TRAIN, random_state=RANDOM_STATE)

cleaning = pd.DataFrame({
    "step": ["Raw rows", "After cleaning + within-file duplicate removal", "Test rows also found in train (removed)", "Final modeling rows"],
    "train": [len(raw_tr), len(tr_clean), 0, len(train)],
    "test":  [len(raw_te), len(te_clean) + int(overlap_mask.sum()), int(overlap_mask.sum()), len(test)],
})
display(cleaning)
print(f"Dissatisfied share | train: {train.target.mean():.3f}  test: {test.target.mean():.3f}")
print(f"{train['drugName'].nunique():,} drugs and {train['condition'].nunique():,} conditions in the training data")

## 2. Phase 2 Recap, Refined
Phase 2 established four things, re-computed here on the **deduplicated** data: the target is imbalanced, dissatisfaction depends strongly on the condition, sentiment alone is only a moderate signal, and these facts shape the model design (class weights, condition and drug controls, F1 and PR-AUC instead of accuracy alone).

In [ ]:
full = pd.concat([train, test])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
rc = full["rating"].value_counts().sort_index()
ax[0].bar(rc.index, rc.values, color=["#E4572E" if r <= 4 else "#1B998B" for r in rc.index])
ax[0].set(title="Ratings after deduplication (red = dissatisfied, 4 or lower)", xlabel="Rating", ylabel="Reviews")

cond = (train.groupby("condition").agg(n=("target", "size"), rate=("target", "mean"))
        .query("n >= 500").sort_values("n", ascending=False).head(10).sort_values("rate"))
ax[1].barh(cond.index, cond["rate"] * 100, color="#6C7A89")
ax[1].axvline(train.target.mean() * 100, color="#E4572E", ls="--", label="Overall rate")
ax[1].set(title="Dissatisfied share, 10 largest conditions (%)", xlabel="% rated 4 or lower"); ax[1].legend()
plt.tight_layout(); plt.show()

# VADER sentiment alone as a baseline (Phase 2 finding), on up to 20,000 test reviews
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    sia = SentimentIntensityAnalyzer()
    vs = test.sample(min(20000, len(test)), random_state=RANDOM_STATE)
    vscore = vs["review"].map(lambda t: sia.polarity_scores(t)["compound"])
    VADER_AUC = roc_auc_score(vs["target"], -vscore)
    VADER_RHO = spearmanr(vscore, vs["rating"])[0]
except ImportError:
    VADER_AUC = VADER_RHO = float("nan")

hi, lo = cond["rate"].idxmax(), cond["rate"].idxmin()
display(Markdown(f"""
- **Imbalance:** {train.target.mean():.1%} of training reviews are dissatisfied, so always predicting "satisfied" is already {1-test.target.mean():.1%} accurate on test. Accuracy alone cannot judge the model.
- **Condition matters:** among the 10 largest conditions, **{hi}** is dissatisfied {cond.loc[hi,'rate']:.1%} of the time and **{lo}** only {cond.loc[lo,'rate']:.1%}. Condition and drug are therefore included as controls, so the text model is not just learning which condition was reviewed.
- **Sentiment alone is moderate:** VADER reaches ROC-AUC **{VADER_AUC:.2f}** (Spearman with rating {VADER_RHO:.2f}). Many dissatisfied patients describe the condition negatively and the drug neutrally, so a general-purpose sentiment tool is a baseline to beat, not the answer.
"""))

## 3. Tool Integration: Claude
**What Claude does here, and what it does not.** The classifier is classical NLP (TF-IDF with logistic regression, plus gradient boosting). It predicts the rating-based label and needs no API calls, so it is cheap to run on every new review. Claude has a supporting role: for a sample of dissatisfied reviews it names the **main reason** (side effects, not effective, cost or access, stopped or switched, other), which gives the stakeholder context the classifier cannot. Claude labels are **never used as training labels or features**, so the classifier cannot learn Claude's mistakes.

### 3.1 Prompt

In [ ]:
REASONS = ["side_effects", "not_effective", "cost_or_access", "stopped_or_switched", "other"]
SYSTEM_PROMPT = """You read one patient review of a medication that the patient rated 4 out of 10 or lower.
Name the single MAIN reason the patient is dissatisfied. Choose exactly one label:
- side_effects: the patient describes adverse effects or physical or mental symptoms caused by the drug
- not_effective: the drug did not work or did not help the condition
- cost_or_access: price, insurance, availability, or the hassle of getting or taking it
- stopped_or_switched: the main point is that they quit or changed drugs, with no clearer cause above
- other: none of the above, or the review is unclear
If several apply, pick the one the patient emphasizes most. Do not guess beyond the text.
Reply with only a JSON object and no markdown: {"reason": "<one label>", "evidence": "<exact quote of at most 12 words from the review>"}"""
print(f"Prompt version {PROMPT_VERSION}, model {CLAUDE_MODEL}\n")
print(SYSTEM_PROMPT)

### 3.2 Labeling run (cached)
Labels are saved to `data/claude_labels.csv` and token usage to `data/claude_usage.csv`, so re-running the notebook makes **no API calls** unless `RELABEL = True`. Replies that are not valid JSON with an allowed label are retried once and counted as invalid if they still fail (Phase 2's pilot returned invalid JSON on most replies, so this is checked explicitly).

In [ ]:
LABELS_CSV, USAGE_CSV, HAND_CSV = DATA / "claude_labels.csv", DATA / "claude_usage.csv", DATA / "handcheck.csv"
ID = "uniqueID" if "uniqueID" in test.columns else None
pool = test[test["target"] == 1]
label_sample = pool.sample(min(N_LABEL, len(pool)), random_state=RANDOM_STATE)

def parse_reply(text):
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m: return None
    try: d = json.loads(m.group(0))
    except Exception: return None
    return d if isinstance(d, dict) and d.get("reason") in REASONS else None

def run_claude_labeling(df):
    import anthropic
    client = anthropic.Anthropic()                      # reads ANTHROPIC_API_KEY from the environment
    rows, usage = [], []
    for i, (idx, r) in enumerate(df.iterrows()):
        rid = r[ID] if ID else idx
        for attempt in (1, 2):
            resp = client.messages.create(model=CLAUDE_MODEL, max_tokens=150, system=SYSTEM_PROMPT,
                                          messages=[{"role": "user", "content": r["review"][:3000]}])
            text = resp.content[0].text
            usage.append(dict(review_id=rid, attempt=attempt, input_tokens=resp.usage.input_tokens,
                              output_tokens=resp.usage.output_tokens))
            parsed = parse_reply(text)
            if parsed: break
        rows.append(dict(review_id=rid, condition=r["condition"], drugName=r["drugName"], review=r["review"],
                         reason=parsed["reason"] if parsed else None,
                         evidence=parsed.get("evidence") if parsed else None,
                         valid=bool(parsed), raw_reply=text, prompt_version=PROMPT_VERSION, model=CLAUDE_MODEL))
        if (i + 1) % 25 == 0: print(f"  labeled {i+1}/{len(df)}")
    pd.DataFrame(rows).to_csv(LABELS_CSV, index=False); pd.DataFrame(usage).to_csv(USAGE_CSV, index=False)

if RELABEL or not LABELS_CSV.exists():
    if os.environ.get("ANTHROPIC_API_KEY"):
        run_claude_labeling(label_sample)
    else:
        print("No saved labels and no ANTHROPIC_API_KEY found. Add the key to .env (README step 3) and re-run this cell;")
        print("the classifier results below do not depend on Claude.")
labels = pd.read_csv(LABELS_CSV) if LABELS_CSV.exists() else None
CLAUDE_OK = labels is not None
if CLAUDE_OK:
    print(f"{len(labels)} reviews labeled | invalid replies after retry: {(~labels['valid']).sum()}")

### 3.3 Sample outputs

In [ ]:
if CLAUDE_OK:
    show = labels[labels["valid"]].head(6).copy()
    show["review"] = show["review"].str.slice(0, 160) + "..."
    display(show[["condition", "review", "reason", "evidence"]])
    rd = labels.loc[labels["valid"], "reason"].value_counts(normalize=True)
    ax = (rd * 100).sort_values().plot.barh(color="#E4572E", figsize=(7, 3), title="Claude-labeled main reason (% of labeled reviews)")
    plt.tight_layout(); plt.show()
else:
    print("Run the labeling cell first.")

### 3.4 Spot-check: hand-check about 30 random reviews
The cell below writes `data/handcheck.csv` with 30 randomly chosen reviews **without Claude's label** (so the hand-label is not anchored). Read each review, type the label you would give in `my_reason` using the same five labels, save the file, and re-run the cell. Agreement is computed against Claude's current labels, so if agreement is low you can revise the prompt (bump `PROMPT_VERSION`, set `RELABEL = True`), re-run, and the same hand labels are re-scored without redoing them.

In [ ]:
AGREE_CSV = OUT / "handcheck_agreement.csv"
agreement = None
if CLAUDE_OK:
    valid = labels[labels["valid"]]
    if not HAND_CSV.exists():
        hc = valid.sample(min(N_HANDCHECK, len(valid)), random_state=RANDOM_STATE)[["review_id", "condition", "review"]].copy()
        hc["my_reason"] = ""
        hc.to_csv(HAND_CSV, index=False)
        print(f"Wrote {HAND_CSV.name} with {len(hc)} reviews. Fill the my_reason column with one of: {REASONS}, then re-run.")
    else:
        hand = pd.read_csv(HAND_CSV)
        hand["my_reason"] = hand["my_reason"].fillna("").astype(str).str.strip()
        done = hand[hand["my_reason"] != ""].merge(valid[["review_id", "reason"]], on="review_id", how="inner")
        if len(done) < len(hand):
            print(f"{len(hand) - len(done)} of {len(hand)} rows still need a my_reason label (or were not re-labeled validly).")
        if len(done):
            agreement = (done["my_reason"] == done["reason"]).mean()
            kappa = cohen_kappa_score(done["my_reason"], done["reason"])
            display(Markdown(f"**Agreement with Claude: {agreement:.0%} on {len(done)} random reviews (Cohen's kappa {kappa:.2f}), prompt {PROMPT_VERSION}.**"))
            display(pd.crosstab(done["my_reason"], done["reason"], rownames=["my label"], colnames=["Claude label"]))
            dis = done[done["my_reason"] != done["reason"]]
            if len(dis): display(dis[["my_reason", "reason", "review"]].assign(review=lambda d: d["review"].str.slice(0, 200)))
            row = pd.DataFrame([dict(prompt_version=PROMPT_VERSION, n=len(done), agreement=agreement, kappa=kappa)])
            prev = pd.read_csv(AGREE_CSV) if AGREE_CSV.exists() else pd.DataFrame()
            pd.concat([prev[prev.get("prompt_version", pd.Series(dtype=str)) != PROMPT_VERSION] if len(prev) else prev, row]).to_csv(AGREE_CSV, index=False)
            if agreement < 0.80:
                display(Markdown("> Agreement is below 80%. Revise the prompt (look at the disagreements above), set `PROMPT_VERSION = 'v2'` and `RELABEL = True`, and re-run before relying on the reason labels."))
else:
    print("Run the labeling cell first.")

### 3.5 API usage and cost

In [ ]:
CLAUDE_COST = None
if CLAUDE_OK and USAGE_CSV.exists():
    u = pd.read_csv(USAGE_CSV)
    tin, tout = u["input_tokens"].sum(), u["output_tokens"].sum()
    CLAUDE_COST = tin / 1e6 * PRICE_IN + tout / 1e6 * PRICE_OUT
    n_rev = u["review_id"].nunique()
    per1k = CLAUDE_COST / max(n_rev, 1) * 1000
    all_dis = int(pd.concat([train, test])["target"].sum())
    display(pd.DataFrame({
        "metric": ["API calls (incl. retries)", "Input tokens", "Output tokens", "Total cost (USD)", "Cost per 1,000 reviews (USD)",
                   "Est. cost to label every dissatisfied review (USD)"],
        "value": [f"{len(u):,}", f"{tin:,}", f"{tout:,}", f"${CLAUDE_COST:.4f}", f"${per1k:.3f}", f"${per1k/1000*all_dis:,.2f} ({all_dis:,} reviews)"]}))
    print(f"Pricing used: ${PRICE_IN}/M input, ${PRICE_OUT}/M output ({CLAUDE_MODEL}, prompts up to 100K tokens). Check the current Anthropic pricing page.")
else:
    print("No usage file yet. It is written when the labeling cell calls the API.")

### 3.6 Trade-offs: when Claude was worth it
- **Worth it:** naming the main *reason* behind a dissatisfied review. Phase 2's rule-based sentence tagger left over half of sentences as "other" and made visible errors, and there is no ready-made reason label in this dataset. A small model handles that nuance at a cost of a few cents per thousand reviews (section 3.5), and the output is easy to hand-check.
- **Not needed:** the prediction itself. The rating already provides the label, and TF-IDF with logistic regression runs on every new review for free and shows which words drive each prediction, which an API call per review would not.
- **Cheaper or simpler options:** (1) label only the reviews the manager actually opens, not the whole dataset; (2) use the Batch API for non-urgent labeling at a lower rate; (3) a keyword rule for the easy categories such as cost words, reserving Claude for the ambiguous rest; (4) shorter prompts and a short `evidence` field to keep output tokens low.
- **Risk managed:** reviews are sent to an external API, so only review text (no IDs, names or dates) is sent, and Claude labels are used for context only, never as classifier training data.

## 4. Modeling (Path A)
**Design.** Features: TF-IDF of the review text (unigrams and bigrams) plus controls (condition, drug, year, review length, exclamation count, capitalization ratio, log of `usefulCount`). Models: logistic regression (interpretable, class-weighted) and gradient boosting on an SVD-reduced TF-IDF (non-linear). Ablations show what text and controls each add, and `usefulCount` (only known after a review is posted, so a possible leakage proxy) is dropped in a separate run. **Model 1 is the primary model, fixed in advance, not picked by test score.** Everything is scored on the untouched test file; the decision threshold is 0.5.

In [ ]:
numeric, categorical = feature_lists(drop_useful=False)
numeric_nu, _ = feature_lists(drop_useful=True)
cols, cols_nu = ["review"] + numeric + categorical, ["review"] + numeric_nu + categorical
y_tr, y_te = train["target"], test["target"]

specs = {   # name -> (pipeline, columns used)
    "Majority-class baseline":            (DummyClassifier(strategy="most_frequent"), cols),
    "LR controls only":                   (build_lr(numeric, categorical, use_text=False), cols),
    "LR text only":                       (build_lr(numeric, categorical, use_controls=False), cols),
    "Model 1: LR text + controls":        (build_lr(numeric, categorical), cols),
    "Model 1 without usefulCount":        (build_lr(numeric_nu, categorical), cols_nu),
    "Model 2: GBM svd-text + controls":   (build_gbm(numeric, categorical), cols),
}
fitted, rows = {}, []
for name, (model, use) in specs.items():
    t0 = time.time(); model.fit(train[use], y_tr); fitted[name] = (model, use)
    for split, X, y in (("train", train[use], y_tr), ("test", test[use], y_te)):
        r = evaluate(name, model, X, y); r["split"] = split; rows.append(r)
    print(f"{name:36s} fit + scored in {time.time()-t0:5.0f}s")
res = pd.DataFrame(rows)
res_te = res[res.split == "test"].set_index("model")
res_tr = res[res.split == "train"].set_index("model")
res.to_csv(OUT / "model_comparison_all_splits.csv", index=False)

### 4.1 Test-set results against the majority-class baseline

In [ ]:
metrics = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
display(res_te[metrics].round(3))
BASE_ACC, BASE_RATE = res_te.loc["Majority-class baseline", "accuracy"], y_te.mean()
print(f"Majority-class baseline: accuracy {BASE_ACC:.3f}, F1 {res_te.loc['Majority-class baseline','f1']:.3f}. "
      f"A random ranker's PR-AUC equals the dissatisfied base rate, {BASE_RATE:.3f}.")

### 4.2 Training vs. testing scores, side by side

In [ ]:
side = pd.concat({"train": res_tr[["accuracy", "f1", "pr_auc", "roc_auc"]], "test": res_te[["accuracy", "f1", "pr_auc", "roc_auc"]]}, axis=1)
side[("gap", "f1 (train - test)")] = res_tr["f1"] - res_te["f1"]
side[("gap", "pr_auc (train - test)")] = res_tr["pr_auc"] - res_te["pr_auc"]
display(side.round(3))

lines = []
for m in side.index:
    if m == "Majority-class baseline": continue
    g, f1te = side.loc[m, ("gap", "pr_auc (train - test)")], res_te.loc[m, "f1"]
    verdict = ("clear overfitting (train far above test)" if g > 0.10 else
               "mild overfitting" if g > 0.04 else "no meaningful overfitting")
    if res_tr.loc[m, "f1"] < 0.45 and g <= 0.04: verdict = "possible underfitting (low on both train and test)"
    lines.append(f"- **{m}**: train-test PR-AUC gap {g:+.3f}, test F1 {f1te:.3f}: {verdict}.")
display(Markdown("**Overfitting / underfitting check (rule of thumb: gap above 0.10 = overfitting, above 0.04 = mild):**\n\n" + "\n".join(lines)))

### 4.3 Confusion matrices and precision-recall curves

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for a, name in zip(ax[:2], ["Model 1: LR text + controls", "Model 2: GBM svd-text + controls"]):
    r = res_te.loc[name]
    cm = np.array([[r.tn, r.fp], [r.fn, r.tp]])
    sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=a,
                xticklabels=["Pred satisfied", "Pred dissatisfied"], yticklabels=["Actual satisfied", "Actual dissatisfied"])
    a.set_title(f"{name}\n(test, threshold 0.5)")
for name in ["LR controls only", "LR text only", "Model 1: LR text + controls", "Model 2: GBM svd-text + controls"]:
    m, use = fitted[name]; p = m.predict_proba(test[use])[:, 1]
    pr, rc, _ = precision_recall_curve(y_te, p); ax[2].plot(rc, pr, label=f"{name} ({res_te.loc[name,'pr_auc']:.2f})")
ax[2].axhline(BASE_RATE, color="grey", ls="--", label=f"Baseline rate ({BASE_RATE:.2f})")
ax[2].set(title="Precision-recall (test)", xlabel="Recall", ylabel="Precision"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

### 4.4 Which features matter (Model 1 coefficients)
Logistic regression coefficients are on standardized or TF-IDF-scaled inputs, so they rank features within the model. They are **associations with the label, not causes.**

In [ ]:
m1, _ = fitted["Model 1: LR text + controls"]
coefs = lr_coefficients(m1)
words = coefs[coefs["feature"].str.startswith("text__")].assign(feature=lambda d: d["feature"].str.replace("text__", "", regex=False))
ctrl = coefs[~coefs["feature"].str.startswith("text__")].assign(feature=lambda d: d["feature"].str.replace(r"^(num|cat)__", "", regex=True))
top_dis, top_sat = words.sort_values("coef", ascending=False).head(15), words.sort_values("coef").head(15)

fig, ax = plt.subplots(1, 3, figsize=(17, 5))
ax[0].barh(top_dis["feature"][::-1], top_dis["coef"][::-1], color="#E4572E"); ax[0].set_title("Words pushing toward DISSATISFIED")
ax[1].barh(top_sat["feature"][::-1], top_sat["coef"][::-1], color="#1B998B"); ax[1].set_title("Words pushing toward SATISFIED")
c10 = ctrl.head(12); ax[2].barh(c10["feature"][::-1], c10["coef"][::-1], color=["#E4572E" if v > 0 else "#1B998B" for v in c10["coef"][::-1]])
ax[2].set_title("Largest control-variable coefficients")
plt.tight_layout(); plt.show()
words.to_csv(OUT / "lr_word_coefficients.csv", index=False)

In [ ]:
best_gap = res_te.loc["Model 1: LR text + controls", "pr_auc"] - res_te.loc["LR controls only", "pr_auc"]
gbm_vs_lr = res_te.loc["Model 2: GBM svd-text + controls", "pr_auc"] - res_te.loc["Model 1: LR text + controls", "pr_auc"]
nu_gap = res_te.loc["Model 1: LR text + controls", "pr_auc"] - res_te.loc["Model 1 without usefulCount", "pr_auc"]
poor = res_te.loc["Model 1: LR text + controls", "f1"] < 0.60
display(Markdown(f"""
**Reading the results**
- **Versus the baseline:** Model 1 reaches F1 {res_te.loc['Model 1: LR text + controls','f1']:.2f} and PR-AUC {res_te.loc['Model 1: LR text + controls','pr_auc']:.2f}, where the majority-class baseline has F1 0.00 and a random ranking has PR-AUC {BASE_RATE:.2f}. Its accuracy ({res_te.loc['Model 1: LR text + controls','accuracy']:.2f}) is {'below the baseline accuracy of ' + format(BASE_ACC, '.2f') + ' because the model is class-weighted to catch dissatisfied reviews and accepts some false alarms; that is why F1 and PR-AUC are the headline metrics' if res_te.loc['Model 1: LR text + controls','accuracy'] < BASE_ACC else 'above the baseline accuracy of ' + format(BASE_ACC, '.2f') + ', but F1 and PR-AUC remain the headline metrics because accuracy hides the minority class'}.
- **What text adds:** text + controls beats controls only by {best_gap:+.3f} PR-AUC.
- **Linear vs. non-linear:** gradient boosting differs from logistic regression by {gbm_vs_lr:+.3f} PR-AUC, so {'the extra complexity is not clearly worth it' if gbm_vs_lr < 0.02 else 'non-linear structure adds real signal'}.
- **Leakage check:** dropping `usefulCount` changes PR-AUC by {-nu_gap:+.3f}, so {'results do not depend on this post-posting field' if abs(nu_gap) < 0.01 else 'the model leans on usefulCount, and the without-usefulCount run is the fairer estimate'}.
- **{'Why performance is only moderate' if poor else 'Remaining errors'}:** ratings are self-reported and noisy, a 4 and a 5 can read the same, many reviews mix a good drug experience with a bad condition, and a bag-of-words model cannot read sarcasm or context. These limit how high precision and recall can go for any text model on this label.
"""))

## 5. Key Findings

In [ ]:
p1 = "Model 1: LR text + controls"
m1, use1 = fitted[p1]
proba = m1.predict_proba(test[use1])[:, 1]; pred = (proba >= 0.5).astype(int)
tmp = test[["condition", "target"]].copy(); tmp["pred"] = pred
g = tmp.groupby("condition").agg(n=("target", "size"), actual_rate=("target", "mean"), flagged_rate=("pred", "mean"))
g["recall"] = tmp[tmp.target == 1].groupby("condition")["pred"].mean()
g["precision"] = tmp[tmp.pred == 1].groupby("condition")["target"].mean()
top_g = g.sort_values("n", ascending=False).head(10)

fig, ax = plt.subplots(2, 2, figsize=(14, 9))
order = ["Majority-class baseline", "LR controls only", "LR text only", p1, "Model 2: GBM svd-text + controls"]
ax[0, 0].barh([o.replace("Model 1: ", "").replace("Model 2: ", "") for o in order][::-1], res_te.loc[order, "pr_auc"][::-1], color="#3E7CB1")
ax[0, 0].axvline(BASE_RATE, color="grey", ls="--"); ax[0, 0].set_title("Finding 1: text adds signal beyond drug and condition (test PR-AUC)")
ax[0, 1].barh(top_dis["feature"][:10][::-1], top_dis["coef"][:10][::-1], color="#E4572E"); ax[0, 1].set_title("Finding 2: words most tied to dissatisfaction")
td = top_g.sort_values("recall")
ax[1, 0].barh(td.index, td["recall"], color="#6C7A89"); ax[1, 0].set_title("Finding 3: recall of dissatisfied reviews by condition"); ax[1, 0].set_xlim(0, 1)
if CLAUDE_OK:
    rd = labels.loc[labels["valid"], "reason"].value_counts(normalize=True).sort_values()
    ax[1, 1].barh(rd.index, rd.values * 100, color="#E4572E"); ax[1, 1].set_title("Finding 4: Claude-labeled main reason (%)")
else:
    ax[1, 1].axis("off"); ax[1, 1].text(0.1, 0.5, "Run the Claude labeling cell to fill this panel", fontsize=12)
plt.tight_layout(); plt.show()

hi_r, lo_r = td["recall"].idxmax(), td["recall"].idxmin()
reason_line = ""
if CLAUDE_OK:
    rd2 = labels.loc[labels["valid"], "reason"].value_counts(normalize=True)
    reason_line = (f"\n4. **Claude gives the queue context.** Among {len(labels)} sampled dissatisfied reviews, the most common main reason is "
                   f"*{rd2.index[0]}* ({rd2.iloc[0]:.0%}), followed by *{rd2.index[1]}* ({rd2.iloc[1]:.0%})"
                   + (f"; hand-check agreement was {agreement:.0%}." if agreement is not None else "; the hand-check is still to be filled in."))
display(Markdown(f"""
1. **The text classifier {'clearly beats' if res_te.loc[p1,'pr_auc'] - BASE_RATE > 0.15 else 'beats'} the baselines.** Model 1 reaches PR-AUC {res_te.loc[p1,'pr_auc']:.2f} (random = {BASE_RATE:.2f}) and F1 {res_te.loc[p1,'f1']:.2f}; the majority-class baseline scores F1 0.00. Text adds {best_gap:+.3f} PR-AUC over drug and condition alone.
2. **Dissatisfaction has a recognizable vocabulary.** The strongest dissatisfied-side words and phrases are: *{', '.join(top_dis['feature'].head(8))}*. The strongest satisfied-side ones are: *{', '.join(top_sat['feature'].head(8))}*.
3. **Performance is uneven across conditions.** Recall of dissatisfied reviews ranges from {td.loc[lo_r,'recall']:.0%} ({lo_r}) to {td.loc[hi_r,'recall']:.0%} ({hi_r}) among the 10 largest conditions.{reason_line}
"""))

## 6. Business Implications and Recommendations
**How it would be used.** Score each incoming review, sort by predicted probability, and let the patient-experience team work the queue from the top. The table shows what that buys compared with reading reviews at random.

In [ ]:
order_idx = np.argsort(-proba); y_sorted = y_te.values[order_idx]
rows = []
for frac in (0.01, 0.05, 0.10, 0.20):
    k = int(len(y_sorted) * frac); top = y_sorted[:k]
    rows.append({"read the top": f"{frac:.0%}", "reviews": k, "share truly dissatisfied": top.mean(),
                 "lift vs. random": top.mean() / BASE_RATE, "share of ALL dissatisfied reviews captured": top.sum() / y_sorted.sum()})
queue = pd.DataFrame(rows); display(queue.round(3))
q5, q10 = queue.iloc[1], queue.iloc[2]
display(Markdown(f"""
**Recommendations (quantified and segmented)**
1. **Work the queue from the top, in tiers.** Reading the top 5% of reviews ({int(q5['reviews']):,} of {len(y_sorted):,} test reviews) finds a dissatisfied patient {q5['share truly dissatisfied']:.0%} of the time, {q5['lift vs. random']:.1f}x better than random, while the top 10% captures {q10['share of ALL dissatisfied reviews captured']:.0%} of all dissatisfied reviews. Staff the first tier for fast outreach; treat deeper tiers as batch review.
2. **Use condition-aware thresholds.** Model recall and flag rates differ by condition (section 5 and 7), so set thresholds per major condition rather than one global cutoff, and re-check them quarterly.
3. **Pair each flagged review with its reason.** Show the Claude-labeled reason (side effect, not effective, cost, stopped) next to the flag so pharmacists and safety staff can route cases without opening every review; start with side-effect cases for safety escalation.
4. **Measure it live.** For the first two months, have staff mark each flagged review as correct or not, and retrain when precision in the top tier drops below the level shown above.

**Limitations and risks specific to this data and method**
- **Self-selected reviewers.** Drugs.com reviewers are people motivated to post, so results describe those reviewers, not all patients taking a drug.
- **Rating is a noisy proxy.** The label is a self-reported star rating cutoff at 4, not a verified unhappy-patient outcome; some identical texts carry different ratings.
- **Duplicates and leakage.** The raw data repeat many review texts across train and test. Removing them lowers headline scores but makes them honest, and the production score should be expected to look like this deduplicated one.
- **Old data.** Reviews span roughly 2008 to 2017, so wording and drug mix may have shifted.
- **A flag is not a diagnosis.** The model ranks text; it does not detect adverse events or judge medical truth.
"""))

## 7. Ethical and Privacy Considerations
**Personal information.** The dataset has no names, usernames or account IDs, only the free-text review, drug, condition, rating, date and a helpfulness count. Patients can still type identifying details into free text, so the scan below counts common patterns. The model uses TF-IDF with `min_df=5`, which drops one-off tokens such as a stray name, and only review text (no IDs, names or dates) is ever sent to the Claude API.

In [ ]:
PII = {
    "email address": r"[\w.+-]+@[\w-]+\.[\w.-]+",
    "phone number": r"\(?\b\d{3}\)?[-. ]\d{3}[-. ]\d{4}\b",
    "URL": r"https?://\S+|www\.\S+",
    "'my name is'": r"\b[Mm]y name is\b",
    "self-stated age ('I am 34', '34 year old')": r"\b(?:I am|I'm|Im|I\u2019m)\s\d{2}\b|\b\d{2}[- ]?(?:year[- ]old|yo|y/o)\b",
}
txt = full["review"]
pii = pd.DataFrame({"pattern": list(PII), "reviews matching": [int(txt.str.contains(p, regex=True).sum()) for p in PII.values()]})
pii["per 10,000 reviews"] = (pii["reviews matching"] / len(txt) * 10000).round(1)
display(pii)
print("Names are not caught by regex; a name-entity pass would be the next step before sharing review text more widely.")

**Under-represented groups.** The data have no patient demographics, so fairness across age, sex or race cannot be measured directly. What can be measured is coverage: conditions and drugs with few reviews are under-represented, so the model has less to learn from for the patients who take them. The table checks whether performance drops there.

In [ ]:
cc = train["condition"].value_counts()
rare = set(cc[cc < 100].index)
is_rare = test["condition"].isin(rare)
def rr(mask):
    t = test[mask]; p = pred[mask.values]
    return dict(reviews=int(mask.sum()), actual_rate=t["target"].mean(),
                recall=(p[t["target"].values == 1]).mean() if (t["target"] == 1).any() else np.nan,
                precision=(t["target"].values[p == 1]).mean() if p.sum() else np.nan)
cov = pd.DataFrame({"conditions with 100+ train reviews": rr(~is_rare), "conditions with fewer than 100 train reviews": rr(is_rare)}).T
display(cov.round(3))
display(top_g.round(3))
display(Markdown(f"""
{len(rare):,} of {cc.shape[0]:,} training conditions have fewer than 100 reviews. They hold {is_rare.mean():.1%} of test reviews and the model's recall there is {cov.loc['conditions with fewer than 100 train reviews','recall']:.0%} versus {cov.loc['conditions with 100+ train reviews','recall']:.0%} elsewhere. In the largest conditions, the model flags {top_g['flagged_rate'].max():.0%} of reviews at most and {top_g['flagged_rate'].min():.0%} at least, so review workload would concentrate in a few conditions unless thresholds are set per condition.
"""))

**Who could be harmed by a wrong prediction.**
- **Patients (false negatives):** a dissatisfied patient with a real side-effect problem is never flagged, so a safety signal or a chance to help is missed. This is the costlier error, and it is why recall in the top tiers and in small conditions is monitored, and why the model must **not** be the only route by which safety reports are noticed.
- **Staff and patients (false positives):** a satisfied patient is contacted unnecessarily, wasting pharmacist time and possibly feeling monitored. This is a lower-harm error but it erodes trust in the queue.
- **Drugs and brands:** a systematically higher flag rate for a drug can unfairly lower its perceived performance if flags are treated as outcomes. Flags are triage, not evidence about a drug.
- **Claude labels:** a wrong reason label (for example "not effective" when the patient describes a side effect) could send a case to the wrong team, which is why labels are hand-checked and shown only as context.

## 8. AI Usage Log

In [ ]:
agree_txt = (f"Hand-checked {N_HANDCHECK} random reviews: {agreement:.0%} agreement with Claude (prompt {PROMPT_VERSION})."
             if agreement is not None else "Hand-check not yet filled in (data/handcheck.csv).")
invalid_txt = f"{int((~labels['valid']).sum())} of {len(labels)} replies still invalid after one retry; " if CLAUDE_OK else ""
ai_log = pd.DataFrame([
    dict(tool="Claude API (" + CLAUDE_MODEL + ")", task="Label the main reason behind dissatisfied reviews",
         prompt="System prompt in section 3.1: one of five labels plus a short evidence quote, JSON only",
         output="JSON reason and evidence per review; saved to data/claude_labels.csv",
         decision="Used for context in the queue and Finding 4 only; never as classifier labels or features",
         verification=invalid_txt + agree_txt + " JSON validity is checked in code, and invalid replies are retried once."),
    dict(tool="Claude API (Phase 2 pilot)", task="Pilot: extract reason from 20 reviews",
         prompt="Early prompt asking for a JSON reason per review",
         output="Most replies were not valid JSON (17 of 20 in the logged Phase 2 run)",
         decision="Added a strict allowed-label check, regex extraction of the JSON object and one retry",
         verification="Caught by running json.loads on every reply and counting failures; Claude was wrong about following the format, not about the content."),
    dict(tool="Claude (chat)", task="Review the Phase 1 modeling script drug_review_models.py",
         prompt="Look into my repo and tell me what is there",
         output="Flagged that usefulCount is included by default and that duplicate texts shared by train and test survive per-file deduplication",
         decision="Ran Model 1 with and without usefulCount, and removed test rows whose text appears in train",
         verification=f"Checked in section 1: {int(overlap_mask.sum()):,} test rows matched train text and were removed; section 4 reports the with/without usefulCount runs."),
    dict(tool="Claude (chat)", task="Draft the presentation and speaker script",
         prompt="Do a presentation on my project work using these instructions and rubric",
         output="First draft framed the project as 'explain, not predict', taken from the Phase 2 notebook notes",
         decision="Reframed everything as a classifier, matching the Phase 1 proposal",
         verification="Wrong framing caught by me when reading the draft against my Phase 1 proposal; Claude then rebuilt the deck as a classifier."),
    dict(tool="Claude (chat)", task="Write this report notebook from the Phase 3 instructions and rubric",
         prompt="Look through the instructions and rubric and help me do the final report",
         output="Notebook code and narrative with results computed in code",
         decision="Used the structure and code, with settings in one cell and all numbers computed rather than typed",
         verification="Code was tested end to end on a synthetic dataset with the same columns before use; every number and plot in this notebook comes from running it on the real data."),
])
pd.set_option("display.max_colwidth", 200)
display(ai_log.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}).hide(axis="index"))
ai_log.to_csv(OUT / "ai_usage_log.csv", index=False)

## 9. Save Results and Update the README
Writes the summary tables to `results/` and refreshes the "Main results" block of `README.md` (between the `RESULTS` markers) so the README always matches this run.

In [ ]:
res_te[metrics].round(4).to_csv(OUT / "model_comparison.csv")
queue.round(4).to_csv(OUT / "queue_lift.csv", index=False)
top_g.round(4).to_csv(OUT / "condition_performance.csv")

m1r = res_te.loc[p1]
block = f"""**Test-set results** (deduplicated; {len(train):,} train / {len(test):,} test reviews; {train.target.mean():.1%} dissatisfied)

| Model | Accuracy | F1 | PR-AUC | ROC-AUC |
|---|---|---|---|---|
""" + "\n".join(f"| {n} | {r.accuracy:.3f} | {r.f1:.3f} | {r.pr_auc:.3f} | {r.roc_auc:.3f} |" for n, r in res_te.iterrows()) + f"""

- Text + controls beats controls only by {best_gap:+.3f} PR-AUC; reading the top 5% of the ranked queue finds a dissatisfied patient {q5['share truly dissatisfied']:.0%} of the time ({q5['lift vs. random']:.1f}x random).
- Claude reason labels: {f"{agreement:.0%} agreement on a hand-check of {N_HANDCHECK} random reviews" if agreement is not None else "hand-check pending"}.
- **API cost:** {f"${CLAUDE_COST:.4f} to label {len(labels)} reviews with {CLAUDE_MODEL}" if CLAUDE_COST is not None else "no API calls in this run"}.
"""
readme = ROOT / "README.md"
if readme.exists():
    s = readme.read_text(encoding="utf-8")
    if "<!-- RESULTS:START -->" in s:
        s = re.sub(r"<!-- RESULTS:START -->.*?<!-- RESULTS:END -->", lambda m: "<!-- RESULTS:START -->\n" + block + "\n<!-- RESULTS:END -->", s, flags=re.S)
        readme.write_text(s, encoding="utf-8"); print("README results block updated.")
    else:
        print("README has no RESULTS markers; paste results/ summary by hand.")
display(Markdown(block))